# Линейная регрессия доходностей облигаций правительства США

## Подготовка данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
dgs1 = pd.read_csv('DGS1.csv')
dgs3 = pd.read_csv('DGS3.csv')

In [ ]:
dgs1.dtypes

In [ ]:
dgs3.dtypes

In [ ]:
merged_df = pd.merge(dgs1, dgs3, on='observation_date', how='outer')

In [ ]:
merged_df = merged_df.loc[merged_df['observation_date'] <= '2009-04-10']

In [ ]:
merged_df['DGS1'] = merged_df['DGS1'].interpolate()
merged_df['DGS3'] = merged_df['DGS3'].interpolate()

In [ ]:
print(len(merged_df))

In [ ]:
dates = pd.to_datetime(merged_df['observation_date'])

In [ ]:
#plt.title('Доходность облигаций правительства США')
plt.plot(dates, merged_df['DGS1'], label='облигации со сроком погашения 1 год' , linestyle='-', linewidth=0.5)
plt.plot(dates, merged_df['DGS3'], label='облигации со сроком погашения 3 года', linestyle='-', linewidth=0.5)
plt.legend()
plt.savefig('source_data_plot.png', dpi=500)

In [ ]:
X = merged_df['DGS1'].to_numpy()
Y = merged_df['DGS3'].to_numpy()

In [ ]:
X_diff = np.diff(X)
Y_diff = np.diff(Y)

## Тест Дики --- Фуллера (черновик)

In [ ]:
from statsmodels.tsa.stattools import adfuller, acf

import linreg
import stattests

m = linreg.OLSModelAdapterFactory()(X, Y)
E = m(X) - Y

print(len(E))

plt.plot(dates, E)
plt.show()

acf_values = acf(E)
plt.stem(np.arange(acf_values.shape[0]), acf_values)
plt.show()

print(
    stattests.augmented_Dickey_Fuller_test(
        E,
        trend_type=stattests.TrendType.LINEAR,
        #auto_lag_IC=stattests.AutoLagInfoCriterionType.AIC
    ).to_str_evaluated()
)

In [ ]:
adfstat, pvalue, usedlag, nobs, critvalues, icbest = adfuller(E, regression='ct')
print(f'{adfstat=}')
print(f'{pvalue=}')
print(f'{usedlag=}')
print(f'{nobs=}')
print(f'{critvalues=}')
print(f'{icbest=}')

## Причинность по Грейнджеру

In [ ]:
import linreg

from stattests import Granger_causality_robust_Wald_test


Granger_causality_kwargs = {
    'XY'     : { 'reason_series': X     , 'effect_series': Y      },
    'YX'     : { 'reason_series': Y     , 'effect_series': X      },
    'XY_diff': { 'reason_series': X_diff, 'effect_series': Y_diff },
    'YX_diff': { 'reason_series': Y_diff, 'effect_series': X_diff }
}

model_adapter_factories = {
    'OLS'  : linreg.OLSModelAdapterFactory(),
    'LAD'  : linreg.LADModelAdapterFactory(),
    'Log'  : linreg.LogModelAdapterFactory(),
    'Huber': linreg.HuberModelAdapterFactory(),
    'Tukey': linreg.TukeyModelAdapterFactory()
}

Granger_causality_test_results = {
    f'({kwargs_key}, {loss_key})':
        Granger_causality_robust_Wald_test(
            **kwargs,
            reason_max_lag=5,
            effect_max_lag=5,
            model_adapter_factory=factory,
            #use_f=False
        )
    for kwargs_key, kwargs in Granger_causality_kwargs.items()
        for loss_key, factory in model_adapter_factories.items()
}

In [ ]:
#with np.printoptions(precision=3):
for k, w in Granger_causality_test_results.items():
    print(f'{k}:\n{w.to_str_evaluated(indentation=1)}')

In [ ]:
gtr = Granger_causality_robust_Wald_test(
    reason_series=X,
    effect_series=Y,
    reason_max_lag=5,
    effect_max_lag=5,
    model_adapter_factory=linreg.OLSModelAdapterFactory(),
    #use_f=False
)

display(gtr.distribution)
display(gtr.statistic)
display(gtr.pvalue)
display(str(gtr))
display(gtr.evaluate())
display(gtr.to_str_evaluated())
print()
print(gtr.to_str_evaluated())

In [ ]:
1/0

## Исследование моделей регрессии

In [ ]:
from statsmodels.tsa.stattools import acf

import djsr
import linreg


def research_models(X, Y, T, directory=None):
    
    datalen = len(X)

    model_adapter_factories = {
        'OLS'  : linreg.OLSModelAdapterFactory(),
        'LAD'  : linreg.LADModelAdapterFactory(),
        'Log'  : linreg.LogModelAdapterFactory(),
        'Huber': linreg.HuberModelAdapterFactory(),
        'Tukey': linreg.TukeyModelAdapterFactory()
    }
    
    model_adapters = {
        key: factory(X, Y)
            for key, factory in model_adapter_factories.items()
    }
    
    for key, model_adapter in model_adapters.items():
        print(f'==== ==== ==== ==== {key} ==== ==== ==== ====')
        display(model_adapter.get_model().summary())
        print('\n')
    
    plt.scatter(X, Y, s=0.25, color='gray')
    for key, model_adapter in model_adapters.items():
        plt.plot(X, model_adapter(X), label=key)
    plt.legend()
    if directory is not None:
        plt.savefig(f'{directory}/regression.png')
    plt.show()
    
    residuals = {
        key: model_adapter(X) - Y
            for key, model_adapter in model_adapters.items()
    }

    for key, residual in residuals.items():
        print(f'{key}: {djsr.DJSR_test(residual)}')

    for key, residual in residuals.items():
        plt.title(key)
        plt.plot(T, residual, linewidth=0.5)
        if directory is not None:
            plt.savefig(f'{directory}/residual_{key}.png')
        plt.show()

    for key, residual in residuals.items():
        plt.title(key)
        acf_values = acf(residual)
        plt.stem(np.arange(acf_values.shape[0]), acf_values)
        if directory is not None:
            plt.savefig(f'{directory}/ACF_{key}.png')
        plt.show()


In [ ]:
research_models(X, Y, dates, directory='results/DGS1_DGS3')

In [ ]:
research_models(Y, X, dates, directory='results/DGS3_DGS1')

In [ ]:
research_models(X_diff, Y_diff, dates[1:], directory='results/DGS1_DGS3_delta')

In [ ]:
research_models(Y_diff, X_diff, dates[1:], directory='results/DGS3_DGS1_delta')